# IT Helpdesk RAG System

**Author:** Faraday Barr Fatahillah


## Overview

This project builds a **Retrieval-Augmented Generation (RAG)** system for an IT
service desk using entirely local components (no paid APIs):

| Component | Technology |
|---|---|
| Ticket classification | TF-IDF + Logistic Regression |
| Vector storage & retrieval | ChromaDB (persistent) |
| Text embeddings | `all-MiniLM-L6-v2` (SentenceTransformer) |
| Answer generation | Ollama (`llama3`) |
| Quality evaluation | LLM-as-Judge (Groundedness / Coherence / Relevance) |

> **Language note:** this notebook is written in English, but the knowledge base
> (`SOP_IT_Helpdesk.md`) and the ticket datasets are in **Indonesian**, modelling
> a real Indonesian enterprise service desk. Keeping the corpus and the queries
> in the same language is what keeps retrieval accurate.

### Workflow
```mermaid
flowchart TB
 subgraph SOURCES["Data Sources"]
        DS["Ticket Data (Training Set)"]
        SOP["SOP IT Helpdesk"]
        DT["Ticket Data (Test Set)"]
  end
 subgraph PREP["Offline Preparation"]
        TRAIN["Classification Model from Training Set (Category)<br>"]
        INDEX["Parsing and Indexing"]
  end
 subgraph STORAGE["Storage"]
        PKL["Save Model"]
        CHROMA["Chroma DB"]
  end
 subgraph PIPELINE["RAG Pipeline"]
        C1["Classify Category (Filtering Questions)"]
        C2["Context Search"]
        C3["Generate Answer from Model"]
        C4["Evaluate Metrics<br>(Groundedness, Relevance, Coherence)<br>"]
  end
    DS --> TRAIN
    SOP --> INDEX
    DT --> C1
    TRAIN --> PKL
    INDEX --> CHROMA
    PKL --> C1
    CHROMA --> C2
    C1 --> C2
    C2 --> C3
    C3 --> C4
```


## Cell 0 - Global Configuration

Defines the constants used throughout the notebook:

- **`OLLAMA_MODEL`** - name of the local LLM served through Ollama (default: `llama3`).
- **`EMBED_MODEL`** - SentenceTransformer model that turns text into embedding vectors.
- **`CHROMA_PATH`** - local directory where ChromaDB persists its vector data.
- **`COLLECTION`** - name of the ChromaDB collection holding the SOP text chunks.
- **`VALID_CATEGORIES`** - the ticket categories the system recognises (`Access`, `Network`, `Hardware`, `ERP`, `Software`, `Other`). These are the same six categories used by the ticket datasets and by the six sections of the SOP.

In [ ]:
OLLAMA_MODEL   = "llama3"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CHROMA_PATH    = "./chroma_db"
COLLECTION     = "helpdesk_tickets"

VALID_CATEGORIES = {"Access", "Network", "Hardware", "ERP", "Software", "Other"}

## Cell 1 - Ticket Classifier Training (TF-IDF + Logistic Regression)

### Goal
Train a text classifier that automatically assigns a category
(Access / Network / Hardware / ERP / Software / Other) to each incoming helpdesk ticket.

### Steps
1. **Load the training data** from `tickets_IT_helpdesk_150each.csv` (150 tickets per category, 900 in total) and the **test data** from `tickets_IT_helpdesk_testset_30each.csv` (30 tickets per category, 180 in total).
2. **Deduplicate** - drop any training row whose issue text also appears in the test set, to prevent data leakage.
3. **Vectorise** - `TfidfVectorizer` with `ngram_range=(1, 2)` so both unigrams and bigrams are captured.
4. **Train** - `LogisticRegression(max_iter=1000)`, chosen for being simple, fast, and strong enough for text classification.
5. **Evaluate** - print a `classification_report` (precision, recall, F1) on the test set.
6. **Save the model** - the `(vectorizer, clf)` pair is pickled to `classifier.pkl` so it can be reloaded without retraining.

### Required files
- `tickets_IT_helpdesk_150each.csv` - training dataset
- `tickets_IT_helpdesk_testset_30each.csv` - test dataset

### Output
- `classifier.pkl` - the trained classification model

In [2]:
import pandas as pd, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df_train = pd.read_csv("tickets_IT_helpdesk_150each.csv")

df_test = pd.read_csv("tickets_IT_helpdesk_testset_30each.csv")

df_train = df_train[~df_train["issue"].isin(df_test["issue"])]

valid_cats = df_train["category"].unique()
df_test = df_test[df_test["category"].isin(valid_cats)]

X_train = df_train["issue"]
y_train = df_train["category"]

X_test = df_test["issue"]
y_test = df_test["category"]

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

print(classification_report(y_test, clf.predict(X_test_vec)))

with open("classifier.pkl", "wb") as f:
    pickle.dump((vectorizer, clf), f)

              precision    recall  f1-score   support

      Access       0.96      0.73      0.83        30
         ERP       0.96      0.83      0.89        30
    Hardware       1.00      0.57      0.72        30
     Network       0.72      0.97      0.83        30
       Other       0.91      1.00      0.95        30
    Software       0.66      0.90      0.76        30

    accuracy                           0.83       180
   macro avg       0.87      0.83      0.83       180
weighted avg       0.87      0.83      0.83       180



## Cell 2 - SOP Parsing and Indexing into ChromaDB

### Goal
Split the IT Helpdesk SOP (Standard Operating Procedure) document into text
*chunks* and index them in ChromaDB as the knowledge base for retrieval.

### How `parse_sop(path)` works
- Reads the SOP Markdown file line by line.
- Each `## SOP-00X` heading marks the start of a new SOP section, which determines the category via `CATEGORY_MAP`.
- Each `### NNN.N` heading marks a single procedure and becomes its own *chunk*.
- A chunk is only closed once the *next* procedure heading is reached, so the parser records which SOP section the open chunk belongs to (`chunk_sop`) separately from the section heading it has most recently seen (`current_sop`). Without that separation, the last procedure of each section inherits the following section's category.
- Every chunk stores: `id`, `title`, `text` (the procedure body), `sop`, and `category`.

### Category mapping
The six SOP sections map onto the same six categories the classifier predicts:
`SOP-001` Access, `SOP-002` Network, `SOP-003` Hardware, `SOP-004` ERP,
`SOP-005` Software, `SOP-006` Other. Because `retrieve()` filters on this
metadata, a wrong mapping here silently sends the wrong SOP text to the LLM.

### ChromaDB configuration
- **`PersistentClient`** - data is stored on disk (`./chroma_db`), so it does not need re-indexing every session.
- **`SentenceTransformerEmbeddingFunction`** - converts text to vectors locally using `all-MiniLM-L6-v2`.
- **`cosine` distance metric** - measures semantic similarity between texts.

### Required files
- `SOP_IT_Helpdesk.md` - the SOP document in Markdown

### Output
- A `helpdesk_tickets` collection in ChromaDB containing every SOP chunk and its metadata.

In [ ]:
import chromadb, re
from chromadb.utils import embedding_functions

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION,
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)


# Mirrors the six SOP sections in SOP_IT_Helpdesk.md, and therefore the six
# ticket categories in the datasets. SOP-004 covers SAP, so it maps to ERP.
CATEGORY_MAP = {
    "SOP-001": "Access",     # Manajemen Akses & Autentikasi
    "SOP-002": "Network",    # Konektivitas Jaringan & VPN
    "SOP-003": "Hardware",   # Prosedur Perangkat Keras
    "SOP-004": "ERP",        # Sistem ERP SAP
    "SOP-005": "Software",   # Email & Office 365
    "SOP-006": "Other",      # Layanan Umum & Kebijakan
}

def parse_sop(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        text = f.read()

    chunks = []
    current_sop = None   # the SOP section heading seen most recently
    chunk_sop = None     # the SOP section the chunk currently being built belongs to
    current_title = None
    current_lines = []

    def flush():
        # Categorise by chunk_sop, not current_sop: by the time a chunk is
        # closed, current_sop may already point at the next SOP section.
        if current_title and current_lines:
            chunks.append({
                "id":       current_title,
                "title":    current_title,
                "text":     "\n".join(current_lines).strip(),
                "sop":      chunk_sop,
                "category": CATEGORY_MAP.get(chunk_sop, "Other"),
            })

    for line in text.splitlines():
        m_sop = re.match(r"^## (SOP-\d+)", line)
        if m_sop:
            current_sop = m_sop.group(1)

        m_proc = re.match(r"^### (\d{3}\.\d+)", line)
        if m_proc:
            flush()
            chunk_sop = current_sop
            current_title = line.lstrip("# ").strip()
            current_lines = [line]
        elif current_title:
            current_lines.append(line)

    flush()

    return chunks

SOP_PATH = "SOP_IT_Helpdesk.md"
sop_chunks = parse_sop(SOP_PATH)

print(f"Parsed {len(sop_chunks)} SOP sections:")
for c in sop_chunks:
    print(f"  [{c['category']:8s}] {c['title']}")

collection.upsert(
    ids       = [c["id"] for c in sop_chunks],
    documents = [c["text"] for c in sop_chunks],
    metadatas = [{"category": c["category"], "sop": c["sop"], "title": c["title"]} for c in sop_chunks],
)

print(f"\nIndexed {collection.count()} SOP chunks into ChromaDB")


## Cell 3 - Reload the Classification Model

Loads the `(vectorizer, clf)` pair previously saved to `classifier.pkl`.
This cell is what you need when re-running the notebook without re-running
Cell 1, so the model does not have to be trained from scratch.


In [4]:
import pickle, ollama

with open("classifier.pkl", "rb") as f:
    vectorizer, clf = pickle.load(f)

## Cell 4 - `classify_query` Function

### Goal
Classify a question/ticket into one of: `Access`, `Network`, `Hardware`, `ERP`, `Software`, or `Other`.

### How it works
1. The query text is vectorised with the trained `vectorizer`.
2. The `clf` model computes a probability for each class.
3. If the highest probability is **below 0.40**, the function returns `"Unknown"` - the model is not confident enough, and the query is not processed further.
4. Otherwise, the highest-scoring class is returned as the category.

### Parameters
| Parameter | Type | Description |
|---|---|---|
| `query` | `str` | The user's question or problem description |

### Returns
- `str` - one of `Access`, `Network`, `Hardware`, `ERP`, `Software`, `Other`, or `Unknown`.

In [ ]:
def classify_query(query: str) -> str:
    """
    Returns one of: Access | Network | Hardware | ERP | Software | Other
    or 'Unknown' when the model confidence is too low.
    """
    vec   = vectorizer.transform([query])
    proba = clf.predict_proba(vec)[0]
    best  = proba.max()

    if best < 0.40:
        return "Unknown"

    return clf.predict(vec)[0]

## Cell 5 - `retrieve` Function

### Goal
Fetch the most relevant SOP chunks from ChromaDB by semantic similarity to the
user's query, restricted to the matching category.

### How it works
1. Queries the ChromaDB collection with a `{"category": category}` filter so retrieval stays within the correct category.
2. ChromaDB computes the cosine distance between the query embedding and each SOP chunk embedding.
3. Returns the `top_k` (default: 3) nearest chunks along with their metadata.

### Parameters
| Parameter | Type | Default | Description |
|---|---|---|---|
| `query` | `str` | - | The user's question text |
| `category` | `str` | - | The classified ticket category |
| `top_k` | `int` | `3` | How many top SOP chunks to retrieve |

### Returns
- `list[dict]` - dictionaries with the keys `issue` (chunk text) and `category`.


In [ ]:
def retrieve(query: str, category: str, top_k: int = 3) -> list[dict]:
    """
    Fetches the top-k most similar SOP procedures from the same category.
    Returns a list of {text, title, category} dicts.
    """
    results = collection.query(
        query_texts = [query],
        n_results   = top_k,
        where       = {"category": category},
    )
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    return [
        {"text": d, "title": m["title"], "category": m["category"]}
        for d, m in zip(docs, metas)
    ]


## Cell 6 - `response_generate` Function

### Goal
The core of the RAG pipeline: combine the retrieved SOP context with the LLM
(Ollama `llama3`) to produce an informative, relevant answer.

### How it works
1. Validates that `category` is in `VALID_CATEGORIES`; if not, returns a default message immediately.
2. Calls `retrieve()` to get the 3 most relevant SOP chunks.
3. Builds a **system prompt** embedding those chunks as reference context for the LLM.
4. Sends the prompt to Ollama with `temperature=0.3` (consistent rather than creative answers) and `num_predict=512` (answer length cap).
5. Returns a dictionary with `query`, `response` (the LLM answer), and `context` (the reference text used).

### Parameters
| Parameter | Type | Description |
|---|---|---|
| `query` | `str` | The user's question or problem description |
| `category` | `str` | The classified category |

### Returns
- `dict` with the keys: `query`, `response`, `context`.


In [ ]:
def response_generate(query: str, category: str) -> dict:
    """
    Returns {query, response, context}.
    Fully local RAG: retrieval from ChromaDB, generation via Ollama.
    """
    if category not in VALID_CATEGORIES:
        return {"query": query, "response": "I don't have that information.", "context": ""}

    hits = retrieve(query, category)
    context_text = "\n\n".join(
        f"[{h['category']}] {h['title']}\n{h['text']}" for h in hits
    )

    system_prompt = (
        "You are a helpful IT helpdesk agent. "
        "Answer the user's question using ONLY the standard operating procedures "
        "below. If they do not cover the question, say so instead of inventing "
        "steps. Be concise and actionable.\n\n"
        "== Relevant SOP procedures ==\n"
        f"{context_text}\n"
        "== End of SOP procedures =="
    )

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": query},
        ],
        options={"temperature": 0.3, "num_predict": 512},
    )

    answer = response["message"]["content"].strip()
    return {"query": query, "response": answer, "context": context_text}


# Smoke test. The corpus is Indonesian, so the queries are too.
# "How do I enable MFA?" -> should classify as Access
print(classify_query("Bagaimana cara mengaktifkan MFA?"))
# "How to cook fried rice" -> out of domain, should return Unknown
print(classify_query("Cara membuat nasi goreng"))    

## Cell 7 - Batch Inference over the Test Set

### Goal
Run the full RAG pipeline (classification -> retrieval -> generation) across every
ticket in the test dataset and persist the results.

### How it works
1. Iterate over every row in `df_test`.
2. For each ticket, `classify_query()` determines its category.
3. Tickets classified as `Unknown` are skipped.
4. `response_generate()` produces an answer using the LLM with SOP context.
5. Results are written to `results.jsonl` with the `jsonlines` library (JSON Lines format - one object per line).

### Output
- `results.jsonl` - a JSON Lines file with one `{query, response, context}` record per ticket.


In [ ]:
import jsonlines

results = []

for _, row in df_test.iterrows():
    query = row["issue"]
    category = classify_query(query)

    if category not in VALID_CATEGORIES:
        continue

    result = response_generate(query, category)
    results.append(result)
    print(f"Done: {row['ticket-id']} | category: {category}")

with jsonlines.open("results.jsonl", mode="w") as writer:
    writer.write_all(results)

print(f"\nSaved {len(results)} records to results.jsonl")

## Cell 8 - LLM-as-Judge Evaluation Functions

### Goal
Evaluate the quality of the answers produced by the RAG pipeline using an LLM as
the judge (*LLM-as-Judge*), across three metrics:

| Metric | Definition |
|---|---|
| **Groundedness** | Is every claim in the answer supported by the provided context? |
| **Coherence** | Is the answer logically structured, fluent, and easy to follow? |
| **Relevance** | Does the answer directly address the user's question? |

Each metric uses a **1-5** scale.

### Function reference

#### `_parse_score(raw: str) -> dict`
Parses the judge's raw LLM output (which may contain non-standard quotes or
Markdown fences) into a `{"score": int, "reason": str}` dictionary, with a regex
fallback for when `json.loads` fails.

#### `evaluate_single(record: dict) -> dict`
Runs all three evaluation prompts against a single RAG result record, returning
the original record enriched with six new fields:
- `eval_groundedness`, `eval_groundedness_reason`
- `eval_coherence`, `eval_coherence_reason`
- `eval_relevance`, `eval_relevance_reason`


In [ ]:
import ollama, json, re

EVAL_PROMPTS = {
    "groundedness": """
You are an evaluation assistant. Rate the GROUNDEDNESS of the response below.

Groundedness measures whether every claim in the response is directly supported
by the provided context. Ignore whether the answer is correct in the real world;
only judge if it is supported by the context.

Scale:
1 - Response contradicts or ignores the context entirely.
2 - Most claims are unsupported or contradict the context.
3 - Some claims are supported; others are not grounded in the context.
4 - Almost all claims are supported by the context.
5 - Every claim is fully and explicitly supported by the context.

Context:
{context}

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
""",

    "coherence": """
You are an evaluation assistant. Rate the COHERENCE of the response below.

Coherence measures whether the response is logically structured, fluent,
and easy to follow — independent of factual accuracy.

Scale:
1 - Incomprehensible or completely disjointed.
2 - Hard to follow; major structural problems.
3 - Understandable but somewhat disjointed or repetitive.
4 - Clear and well-structured with minor issues.
5 - Perfectly clear, logical, and easy to follow.

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
""",

    "relevance": """
You are an evaluation assistant. Rate the RELEVANCE of the response to the query.

Relevance measures whether the response directly addresses what the user asked.

Scale:
1 - Completely off-topic.
2 - Barely related; misses the main point.
3 - Partially addresses the query but misses key aspects.
4 - Mostly addresses the query with minor gaps.
5 - Fully and directly answers the query.

Query:
{query}

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
"""
}

def _parse_score(raw: str) -> dict:
    """Extract {score, reason} from the judge's raw output."""
    clean = re.sub(r"```[\w]*", "", raw).strip()
    clean = clean.replace("\u201c", "\"").replace("\u201d", "\"")
    clean = clean.replace("\u2018", "'").replace("\u2019", "'")
    m_obj = re.search(r"\{[^}]+\}", clean, re.DOTALL)
    if m_obj:
        clean = m_obj.group(0)
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        m = re.search(r"\b([1-5])\b", raw)
        score = int(m.group(1)) if m else 0
        r = re.search(r'reason["\':\\s]+([^,}]+)', raw, re.IGNORECASE)
        reason = r.group(1).strip().strip('"\' ') if r else raw[:120]
        return {"score": score, "reason": reason}


def evaluate_single(record: dict) -> dict:
    """
    Runs all three judges on one result record.
    record must have keys: query, response, context
    Returns the record enriched with eval_groundedness, eval_coherence, eval_relevance.
    """
    scores = {}
    for metric, prompt_template in EVAL_PROMPTS.items():
        prompt = prompt_template.format(
            query    = record.get("query", ""),
            response = record.get("response", ""),
            context  = record.get("context", "(no context)"),
        )
        raw = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0, "num_predict": 128},
        )["message"]["content"].strip()

        parsed = _parse_score(raw)
        scores[f"eval_{metric}"] = parsed.get("score", 0)
        scores[f"eval_{metric}_reason"] = parsed.get("reason", "")

    return {**record, **scores}


print("Evaluator ready.")

Evaluator ready.


## Cell 9 - Batch Evaluation of All RAG Results

### Goal
Load `results.jsonl`, run the LLM-as-Judge evaluation on every record, and save
the scored output.

### How it works
1. Read every record from `results.jsonl`.
2. Call `evaluate_single()` for each one - each call sends 3 requests to Ollama (one per metric).
3. Print scores in real time: `G=` (Groundedness), `C=` (Coherence), `R=` (Relevance).
4. Write all evaluated records to `results_evaluated.jsonl`.

### Output
- `results_evaluated.jsonl` - complete records with the score and reasoning for all three metrics.


In [ ]:
import jsonlines

with jsonlines.open("results.jsonl") as reader:
    rag_results = list(reader)

print(f"Evaluating {len(rag_results)} records...")

evaluated = []
for i, record in enumerate(rag_results):
    scored = evaluate_single(record)
    evaluated.append(scored)
    print(
        f"[{i+1}/{len(rag_results)}] "
        f"G={scored['eval_groundedness']} "
        f"C={scored['eval_coherence']} "
        f"R={scored['eval_relevance']}  "
        f"| {record['query'][:60]}"
    )

with jsonlines.open("results_evaluated.jsonl", mode="w") as writer:
    writer.write_all(evaluated)

print(f"\nSaved to results_evaluated.jsonl")

## Cell 10 - Evaluation Summary Statistics

### Goal
Show aggregate summary statistics from the evaluation, to judge the overall
performance of the RAG system.

### How it works
1. Convert the evaluated records into a pandas `DataFrame`.
2. Compute descriptive statistics (`mean`, `min`, `max`, `std`) for all three metrics.
3. Print each metric's average in `X.XX / 5.00` format.


In [ ]:
import pandas as pd

eval_df = pd.DataFrame(evaluated)

metrics = ["eval_groundedness", "eval_coherence", "eval_relevance"]

summary = eval_df[metrics].agg(["mean", "min", "max", "std"]).round(2)
summary.columns = [m.replace("eval_", "").capitalize() for m in metrics]
print(summary.to_string())

print("\n── Overall averages ──")
for m in metrics:
    label = m.replace("eval_", "").capitalize()
    print(f"{label:15s}: {eval_df[m].mean():.2f} / 5.00")

## Cell 11 - Identifying Problem Records

### Goal
Show, in detail, the tickets that scored poorly on at least one evaluation
metric, for analysis and system improvement.

### How it works
1. Filter `eval_df` for records where **Groundedness**, **Coherence**, or **Relevance** falls below the `THRESHOLD` (default: **3**).
2. Print the query, the first 200 characters of the response, all three scores, and the judge's reasoning for each flagged record.

### How to read the results
- **Low Groundedness** -> the LLM answer contains claims the SOP context does not support; consider improving the prompt or adding SOP chunks.
- **Low Coherence** -> the answer is hard to follow; generation parameters may need tuning.
- **Low Relevance** -> the classifier mislabelled the ticket, or the retrieved context was off-target.


In [ ]:
THRESHOLD = 3

flagged = eval_df[
    (eval_df["eval_groundedness"] < THRESHOLD) |
    (eval_df["eval_coherence"]    < THRESHOLD) |
    (eval_df["eval_relevance"]    < THRESHOLD)
]

print(f"{len(flagged)} records scored below {THRESHOLD} on at least one metric:\n")

for _, row in flagged.iterrows():
    print(f"Query   : {row['query']}")
    print(f"Response: {row['response'][:200]}")
    print(f"Scores  : G={row['eval_groundedness']}  C={row['eval_coherence']}  R={row['eval_relevance']}")
    print(f"Reasons : {row['eval_groundedness_reason']}")
    print("-" * 80)